# Notebook 10 — Avaliação do RAG: baseline, hipóteses e métricas

O RAG dos notebooks 04 a 09 é avaliado como um **experimento**, seguindo o protocolo
do projeto: uma baseline congelada (B0, o pipeline da IA Gen aula 02), hipóteses que
mudam uma variável por vez (H1 a H4), métricas fixadas antes de rodar, um teste
trancado e análise de erro — o roteiro dos notebooks de Deep Learning.

| Seção | Tema | Função em `src/voc_rag` | Origem |
|---|---|---|---|
| 0 | Configuração da rodada | `experimento.ConfigAvaliacao` | — |
| 1 | O gold set | `goldset.carregar_gold` | IA Gen 02 (gabarito) |
| 2 | Os sistemas comparados | `sistemas.criar_sistemas` | protocolo de avaliação |
| 3 | Quanto custa rodar | `experimento.estimar_chamadas` | IA Gen 02 (custo) |
| 4 | Respostas: cache ou ao vivo | `experimento.rodar_sistemas` | — |
| 5 | Visão geral | `experimento.resumo_por_sistema` | — |
| 6 | Recuperação | `acerto@10`, MRR, precisão@10 | IA Gen 02 (recall@k) |
| 7 | Recusa e bloqueio | `experimento.matriz_recusa` | requisito 4 do TC4 |
| 8 | Geração: citações e fidelidade | juiz de fidelidade | IA Gen 02 (LLM-as-a-judge) |
| 9 | Operação: latência, chamadas e tokens | `ContadorLLM` | IA Gen 02 (latência e custo) |
| 10 | Análise de erro | `experimento.comparacao_pareada` | DL aula 06 |
| 11 | Teste trancado | `experimento.bootstrap_ic` | protocolo |

> **Este notebook roda sem chave e sem custo.** Por padrão ele só lê as respostas já
> gravadas em `avaliacao/respostas/` (versionadas no git) e recalcula as métricas.
> Para ver o pipeline rodando de verdade, mude a configuração da Seção 0 para
> `modo="ao_vivo"` com poucas perguntas: aí ele usa o `.env` de quem executa.

In [ ]:
import inspect

import pandas as pd

from voc_rag import experimento, goldset, sistemas
from voc_rag.experimento import ConfigAvaliacao

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)


def mostrar_codigo(objeto):
    """Imprime o código-fonte de uma função ou classe do src/, para lermos junto com o resultado."""
    print(inspect.getsource(objeto))

## Seção 0 — Configuração da rodada

Tudo o que muda o custo está em `ConfigAvaliacao`:

| Campo | Padrão | O que faz |
|---|---|---|
| `modo` | `"cache"` | `"cache"` só lê respostas gravadas (zero chamadas); `"ao_vivo"` executa |
| `sistemas` | os 5 | quais sistemas entram (B0, H1, H2, H3, H4) |
| `particao` | `"dev"` | `"teste"` é trancado: exige `abrir_teste=True` |
| `max_perguntas` | todas | ao vivo, limita as perguntas alternando os estratos |
| `juiz_relevancia` | desligado | rotula o pool de reviews (1 chamada por pergunta) |
| `juiz_fidelidade` | desligado | confere cada afirmação (1 chamada por resposta) |

Exemplo barato para ver ao vivo (alguns centavos):
`ConfigAvaliacao(modo="ao_vivo", max_perguntas=3, sistemas=["B0", "H4"])`.
A rodada ao vivo grava em `artifacts/avaliacao/ao_vivo/`, fora do git, e não mexe no cache.

In [ ]:
CONFIG = ConfigAvaliacao()          # padrão: lê o cache, nenhuma chamada ao LLM
mostrar_codigo(ConfigAvaliacao)
CONFIG

## Seção 1 — O gold set *(IA Gen aula 02: gabarito antes de medir)*

60 perguntas escritas **antes** de olhar os resultados, em 7 estratos — cada um mede um
comportamento que o enunciado cobra. A partição dev/teste foi sorteada dentro de cada
estrato; o teste fica trancado até a rodada final.

As agulhas (B) e paráfrases (C) apontam para uma review real (`review_alvo`): a métrica
é achar essa review no top-10. As paráfrases usam outras palavras de propósito: é ali
que a busca vetorial tem de mostrar valor sobre a lexical.

In [ ]:
print(goldset.__doc__.split("\n\n")[1])
goldset.resumo(goldset.carregar_gold())

In [ ]:
itens = experimento.selecionar_perguntas(CONFIG)
exemplos = pd.DataFrame([i.model_dump() for i in itens]).groupby("estrato").head(1)
exemplos[["id", "estrato", "pergunta", "esperado", "review_alvo", "subtemas"]]

## Seção 2 — Os sistemas comparados *(protocolo: uma variável por vez)*

B0 é o pipeline da aula 02 aplicado às reviews: busca vetorial, top-10 e um prompt que
manda responder só com as evidências e recusar quando não houver. Cada hipótese soma
uma melhoria e diz, **antes de rodar**, em que estrato o ganho deve aparecer:

| Sistema | Muda | Previsão | Estrato |
|---|---|---|---|
| H1 | busca híbrida (BM25 + vetorial, RRF) | ganha em termos exatos | B |
| H2 | + Multi-Query | ganha em paráfrase e perguntas amplas | C, A |
| H3 | + roteador com filtros por metadado | precisão sobe quando a pergunta pede filtro | D |
| H4 | + guardrail e checagem de evidência | recusa certa sobe sem recusar à toa | E, F, G |

De B0 a H3, as citações saem como o LLM escreveu: assim medimos quantas são inválidas.
H4 é o grafo do notebook 09, que valida as citações.

In [ ]:
print(sistemas.PROMPT_SIMPLES.messages[0].prompt.template)
pd.DataFrame({"descrição": sistemas.SISTEMAS, "chamadas por pergunta": sistemas.CHAMADAS_POR_PERGUNTA})

## Seção 3 — Quanto custa rodar *(IA Gen aula 02: latência e custo)*

A estimativa é um **teto**: recusas e bloqueios gastam menos (o H4 bloqueado não chama o
LLM). O juiz de fidelidade só roda nas respostas "respondida". O gasto real fica gravado
por pergunta e aparece na Seção 9.

In [ ]:
experimento.estimar_chamadas(CONFIG, itens)

## Seção 4 — Respostas: do cache ou ao vivo

No modo `"cache"` esta célula não chama nada. No modo `"ao_vivo"` ela carrega a base, o
índice e o LLM, roda só as perguntas que faltam (uma rodada interrompida continua de
onde parou) e, se ligados, os juízes.

In [ ]:
if CONFIG.modo == "ao_vivo":
    from voc_rag import busca, indice, llm
    from voc_rag.preparacao import carregar_base

    documentos = carregar_base()
    indice_vetorial, _ = indice.carregar_indice()
    buscador = busca.Buscador(documentos, indice_vetorial, indice.carregar_modelo())
    modelo_chat = llm.criar_llm()
    experimento.rodar_sistemas(CONFIG, sistemas.criar_sistemas(documentos, buscador, modelo_chat,
                                                                CONFIG.sistemas), itens)
    if CONFIG.juiz_relevancia:
        experimento.rotular_pool(CONFIG, itens, experimento.ler_respostas(CONFIG), documentos, modelo_chat)
    if CONFIG.juiz_fidelidade:
        experimento.julgar_fidelidade(CONFIG, itens, experimento.ler_respostas(CONFIG), documentos, modelo_chat)
else:
    print(f"modo cache: lendo {CONFIG.pasta_saida} (nenhuma chamada ao LLM)")

ids = {i.id for i in itens}
respostas = experimento.ler_respostas(CONFIG).query("id in @ids")
julgamentos = experimento.ler_julgamentos(CONFIG)
fidelidade = experimento.ler_fidelidade(CONFIG)
metricas = experimento.metricas_por_pergunta(respostas, itens, julgamentos, fidelidade)
print(f"{len(respostas)} respostas, {len(julgamentos)} julgamentos de relevância, "
      f"{len(fidelidade)} respostas com fidelidade julgada")

## Seção 5 — Visão geral

Cada métrica é a média só nas perguntas em que ela se aplica (acerto@10 nas agulhas,
filtros nas perguntas D...). `acerto_comportamento` é a mais ampla: respondeu quando
devia, recusou quando não havia evidência, bloqueou o que era perigoso.

In [ ]:
experimento.resumo_por_sistema(metricas)

## Seção 6 — Recuperação *(IA Gen aula 02: se a evidência não vem, nem o melhor LLM salva)*

- **acerto@10 e MRR** (agulhas B e paráfrases C): a review-alvo está no top-10? Em que posição?
- **precisão@10** (A, C, D): fração do top-10 julgada relevante. O julgamento vem do
  *pooling* do protocolo: junta-se o top-10 de todos os sistemas e cada review é
  rotulada uma vez (pré-rótulo do LLM, revisão humana numa amostra).
- **filtros exatos** (D): o roteador extraiu exatamente os filtros esperados?

In [ ]:
for metrica in ["acerto@10", "rr", "precisao@10"]:
    if metrica in metricas:
        print(f"\n{metrica} por estrato")
        print(experimento.por_estrato(metricas, metrica))

In [ ]:
print("H1 contra B0 no MRR das agulhas (previsão: H1 ganha em B):",
      experimento.comparacao_pareada(metricas.query("estrato == 'B'"), "rr", "B0", "H1"))
print("H2 contra H1 no MRR das paráfrases (previsão: H2 ganha em C):",
      experimento.comparacao_pareada(metricas.query("estrato == 'C'"), "rr", "H1", "H2"))

In [ ]:
if "filtros_exatos" in metricas:
    d = respostas.merge(metricas[["sistema", "id", "filtros_exatos"]], on=["sistema", "id"])
    esperados = {i.id: i.filtros.como_kwargs() for i in itens}
    d = d[d["id"].str.startswith("D")].assign(esperado=lambda t: t["id"].map(esperados))
    display(d.pivot_table(index="id", columns="sistema", values="filtros_exatos", aggfunc="first"))

**Calibração do juiz de relevância:** o protocolo pede que um humano revise uma
amostra dos rótulos. A amostra abaixo é sorteada com semente fixa; quem revisar troca
`rotulador` para `"humano"` em `avaliacao/respostas/julgamentos.csv` e corrige `relevante`.

In [ ]:
if len(julgamentos):
    from voc_rag.config import SEMENTE
    amostra = julgamentos.sample(min(20, len(julgamentos)), random_state=SEMENTE)
    print(julgamentos["rotulador"].value_counts().to_dict())
    display(amostra)

## Seção 7 — Recusa e bloqueio *(requisito 4 do TC4)*

A classe positiva é "não respondeu". A célula mais grave é a **alucinação**: responder
uma pergunta sem evidência (E), fora de escopo (F) ou adversarial (G). A previsão de H4
é zerar essa coluna sem aumentar a falsa recusa em A a D.

In [ ]:
experimento.matriz_recusa(metricas)

In [ ]:
experimento.por_estrato(metricas, "acerto_comportamento")

## Seção 8 — Geração: citações e fidelidade *(IA Gen aula 02: LLM-as-a-judge)*

- **citações válidas**: fração dos `review_id` citados que estavam no contexto. De B0 a
  H3 nada filtra a citação; no H4 o código remove id inventado (notebook 08).
- **quantificação sem base**: a resposta diz "a maioria", "muitos" ou um percentual? Com
  10 reviews no contexto, o LLM não tem como saber a frequência na base.
- **fidelidade**: o juiz confere, afirmação por afirmação, se as reviews citadas a
  sustentam. É a parte mais cara (1 chamada por resposta) e por isso pode ser desligada.

In [ ]:
colunas = [c for c in ["citacoes_validas", "quantificacao_sem_base", "fidelidade"] if c in metricas]
metricas.groupby("sistema")[colunas].mean().round(3)

In [ ]:
if len(fidelidade):
    import json
    exemplo = fidelidade.sort_values("fidelidade").iloc[0]
    resposta = respostas.query("sistema == @exemplo.sistema and id == @exemplo.id").iloc[0]
    print(f"[{exemplo.sistema} {exemplo.id}] fidelidade {exemplo.fidelidade:.2f}\n{resposta.resumo}\n")
    for ponto, v in zip(resposta.pontos, json.loads(exemplo.veredictos)):
        print(f"- {ponto['afirmacao']} {ponto['review_ids']}\n    sustentada={v['sustentada']}: {v['motivo']}")

## Seção 9 — Operação: latência, chamadas e tokens

Gasto real, contado por um callback do LangChain em cada chamada. Os tokens de
raciocínio (modelos que "pensam" antes de responder) são cobrados como saída.

In [ ]:
respostas.groupby("sistema").agg(
    perguntas=("id", "count"), chamadas=("chamadas", "sum"),
    tokens_entrada=("tokens_entrada", "sum"), tokens_saida=("tokens_saida", "sum"),
    tokens_raciocinio=("tokens_raciocinio", "sum"),
    latencia_media_s=("latencia_ms", lambda s: round(s.mean() / 1000, 1)),
    latencia_p95_s=("latencia_ms", lambda s: round(s.quantile(0.95) / 1000, 1)),
)

## Seção 10 — Análise de erro *(DL aula 06: três recortes)*

1. O que o sistema final (H4) ainda erra — e em que etapa (status, motivo, avisos).
2. **Regressões**: perguntas que um sistema acertava e o seguinte passou a errar.
3. Perguntas que **todos** erram: o problema pode estar no gold set ou na própria base.

In [ ]:
erros = metricas.query("not acerto_comportamento")
ultimo = CONFIG.sistemas[-1]
(erros.query("sistema == @ultimo")
      .merge(respostas[["sistema", "id", "resumo", "avisos"]], on=["sistema", "id"])
      [["id", "estrato", "esperado", "status", "resumo", "avisos"]])

In [ ]:
ordem = CONFIG.sistemas
for anterior, seguinte in zip(ordem, ordem[1:]):
    placar = experimento.comparacao_pareada(metricas, "acerto_comportamento", anterior, seguinte)
    print(f"{anterior} -> {seguinte}: {placar}")
todos_erram = erros.groupby("id")["sistema"].nunique()
print("\nperguntas que todos os sistemas erram:", sorted(todos_erram[todos_erram == len(ordem)].index))

## Seção 11 — Teste trancado *(protocolo: roda uma vez, no fim)*

Ajustar o sistema olhando o teste invalida o resultado, como em qualquer experimento de
DL. Por isso `ConfigAvaliacao(particao="teste")` dá erro sem `abrir_teste=True`, e o
teste roda **uma única vez**, com a baseline e o sistema final. A célula abaixo só lê o
resultado gravado; com 20 perguntas, o intervalo de confiança por bootstrap mostra o
quanto a diferença é sólida.

In [ ]:
CONFIG_TESTE = ConfigAvaliacao(particao="teste", abrir_teste=True, sistemas=["B0", "H4"])
try:
    itens_teste = experimento.selecionar_perguntas(CONFIG_TESTE)
    respostas_teste = experimento.ler_respostas(CONFIG_TESTE)
    metricas_teste = experimento.metricas_por_pergunta(
        respostas_teste, itens_teste, experimento.ler_julgamentos(CONFIG_TESTE),
        experimento.ler_fidelidade(CONFIG_TESTE))
    display(experimento.bootstrap_ic(metricas_teste, "acerto_comportamento"))
    display(experimento.resumo_por_sistema(metricas_teste))
except FileNotFoundError:
    print("o teste ainda não foi rodado")

## Resumo

- A avaliação compara uma **baseline congelada** com hipóteses que mudam **uma variável
  por vez**, nas mesmas 60 perguntas.
- As respostas ficam em **cache versionado**: a análise é reproduzível sem custo, e o
  modo ao vivo deixa qualquer pessoa rodar uma amostra com a própria chave.
- As métricas cobrem os critérios do enunciado: recuperar (acerto@10, MRR, precisão@10),
  reconhecer falta de informação (matriz de recusa), não inventar (fidelidade,
  quantificação sem base), rastrear (citações válidas) e filtrar por metadado.